In [52]:
import requests
import numpy as np
import pandas as pd
import sqlite3
from datetime import datetime, timezone


Create tables

In [ ]:
conn = sqlite3.connect("screener.db")
cursor = conn.cursor()

cursor.execute("""
    CREATE TABLE IF NOT EXISTS pools (
    pair_address TEXT PRIMARY KEY,
    token_address TEXT,
    chain_id TEXT,
    symbol TEXT,
    dex_id TEXT,
    price_usd REAL,
    liquidity_usd REAL,
    volume_m5 REAL,
    volume_h1 REAL,
    volume_h24 REAL,
    price_change_m5 REAL,
    price_change_h1 REAL,
    price_change_h24 REAL,
    market_cap REAL,
    fdv REAL,
    pair_created_at INTEGER,
    first_seen_at TIMESTAMP,
    last_updated_at TIMESTAMP
)
""")

cursor.execute("""
    CREATE TABLE IF NOT EXISTS pool_snapshots (
        id               INTEGER PRIMARY KEY AUTOINCREMENT,
        pair_address     TEXT REFERENCES pools(pair_address),
        price_usd        REAL,
        liquidity_usd    REAL,
        volume_m5        REAL,
        volume_h1        REAL,
        volume_h24       REAL,
        price_change_m5  REAL,
        price_change_h1  REAL,
        price_change_h24 REAL,
        market_cap       REAL,
        fdv              REAL,
        snapshot_at      TIMESTAMP
    )
""")

In [ ]:
conn.commit()

response = requests.get("https://api.dexscreener.com/token-profiles/latest/v1", headers={"Accept": "*/*"})
tokens = response.json()
all_pairs = []
new_tokens_count = 0

for token in tokens:
    chain_id = token.get("chainId")
    token_address = token.get("tokenAddress")

    cursor.execute(
        """SELECT 1 FROM pools WHERE token_address = ? """,
        (token_address,)
    )

    if cursor.fetchone():
        continue

    print(f"New token discovered: {token_address}")
    new_tokens_count += 1

    now = datetime.now(timezone.utc).isoformat()
    data = requests.get(f"https://api.dexscreener.com/token-pairs/v1/{chain_id}/{token_address}",headers={"Accept": "*/*"})
    pair_response = data.json()

    pair_details = [{ "token_address" : token_address, 
                     "symbol" : item.get('baseToken').get('symbol'),
                     "pair_address" : item.get('pairAddress'),
                     "dex_id" : item.get('dexId'),
                     "price_usd" : item.get('priceUsd'),
                     "price_change_m5" : item.get('priceChange', {}).get('m5'),
                     "price_change_h1" : item.get('priceChange', {}).get('h1'),
                     "price_change_h24" : item.get('priceChange', {}).get('h24'),
                     "liquidity_usd" : item.get('liquidity', {}).get('usd'),
                     "volume_m5" : item.get('volume', {}).get('m5'),
                     "volume_h1" : item.get('volume', {}).get('h1'),
                     "volume_h24" : item.get('volume', {}).get('h24'),
                     "market_cap" : item.get('marketCap'),
                     "fdv" : item.get('fdv'),
                     "pair_created_at" : item.get('pairCreatedAt'),
                     "timestamp_fetched" : now,
    }
    for item in pair_response ]
    all_pairs.extend(pair_details)

    for pool in pair_details:
        cursor.execute(
            """
            INSERT INTO pools (
                pair_address,
                token_address,
                chain_id,
                symbol,
                dex_id,
                price_usd,
                liquidity_usd,
                volume_m5,
                volume_h1,
                volume_h24,
                price_change_m5,
                price_change_h1,
                price_change_h24,
                market_cap,
                fdv,
                pair_created_at,
                first_seen_at,
                last_updated_at
            )
            VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?) ON CONFLICT(pair_address) DO UPDATE SET
                price_usd       = excluded.price_usd,
                liquidity_usd   = excluded.liquidity_usd,
                volume_m5       = excluded.volume_m5,
                volume_h1       = excluded.volume_h1,
                volume_h24      = excluded.volume_h24,
                price_change_m5  = excluded.price_change_m5,
                price_change_h1  = excluded.price_change_h1,
                price_change_h24 = excluded.price_change_h24,
                market_cap      = excluded.market_cap,
                fdv             = excluded.fdv,
                last_updated_at = excluded.last_updated_at
            """,
            (
                pool["pair_address"],
                pool["token_address"],
                chain_id,
                pool["symbol"],
                pool["dex_id"],
                pool["price_usd"],
                pool["liquidity_usd"],
                pool["volume_m5"],
                pool["volume_h1"],
                pool["volume_h24"],
                pool["price_change_m5"],
                pool["price_change_h1"],
                pool["price_change_h24"],
                pool["market_cap"],
                pool["fdv"],
                pool["pair_created_at"],
                now,
                now
            )
        )
        cursor.execute(
            """
            INSERT INTO pool_snapshots (
                pair_address, 
                price_usd, 
                liquidity_usd,
                volume_m5, 
                volume_h1, 
                volume_h24,
                price_change_m5, 
                price_change_h1, 
                price_change_h24,
                market_cap, 
                fdv, snapshot_at
            ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
            """,
            (
                pool["pair_address"], 
                pool["price_usd"], 
                pool["liquidity_usd"],
                pool["volume_m5"], 
                pool["volume_h1"], 
                pool["volume_h24"],
                pool["price_change_m5"], 
                pool["price_change_h1"], 
                pool["price_change_h24"],
                pool["market_cap"], 
                pool["fdv"], now
            )
)
conn.commit()

df = pd.DataFrame(all_pairs)
print(f"New tokens discovered: {new_tokens_count}")
print(f"Pairs collected: {len(df)}")
conn.close()


New token discovered: NpqoucNKKJ62PwikH4NBvTm3R7mgREXoTtv7nJSGLUE
New token discovered: GKTYHhTAsBnLS6W6idMB5qS3cTFoL7tE8654F1bRpump
New token discovered: BGuwU5SdeH93cT3WyN2K4e4vJuqyaFBAkzaZ2wtJUSoH
New token discovered: EhsRtHg5K7tinMp5auPqHpiPfpq146UkSfbj4UCopump
New token discovered: 6CsMmCzJWjYh7cP8rsjNjP4mP5mvAryd6dTsasEQpump
New token discovered: 6ZDiSZ8DkRM5Grhbm8tbJ6x52DdeQtK8gRRJQsfSpump
New token discovered: 3hxFNBokATEkRw9HPkzYv3jm3NL2SWTPRgK5ZNaypump
New token discovered: GveuF8kvKJK2dZu2pN62UzpyhjKjhshzwNGwBC89HWEN
New token discovered: 6KTNucbr3q2wNZpP33pGVUuuTS7MVYw8sq1ym9thwFUa
New token discovered: 25vej2BvAyitbH2nLq4m9BTnvDp5ygMDJJZWpUWNpump
New token discovered: 6MbDfi8uMvyxpUcc1UKPQFpWg6WZpq4szGtg3arFpump
New token discovered: 9XWvURbG8mdPziejQQ8xGtqZy5A4aRwqcJ714DeYpump
New token discovered: FMk6FhiZpK1D6HyP2FFK1RqH4CbR7QQYDvQhxzhjpump
New token discovered: HyH4Tf3fqBNwk9yCwFuGdiE9qmhe93Jok7mQqW3spump
New token discovered: 8uBG3Lzg59Z4EgQZCq8vQLfw37Gen6g9jrnogaMYp